# Exploração dos Dados

In [13]:
import sqlite3
import pandas as pd

## 1. Objetivo

O objetivo deste projeto é desenvolver um **modelo preditivo capaz de estimar o valor total das vendas das próximas seis semanas para cada loja de uma rede de farmácias**.

A previsão será utilizada para apoiar decisões estratégicas, permitindo identificar lojas com maior potencial de retorno e auxiliar no planejamento de investimentos em **reformas, expansão e campanhas de marketing**.

Para cada loja e data de referência, o modelo deverá estimar o **valor total das vendas nos 42 dias seguintes**. Dessa forma, o alvo (*target*) é calculado como a soma das vendas diárias realizadas durante as seis semanas posteriores à data de referência.

Para o desenvolvimento do modelo, estão disponíveis diferentes fontes de informação sobre as lojas:

* **Histórico diário de vendas**, contendo informações sobre vendas, número de clientes, funcionamento da loja, promoções e feriados;
* **Características das lojas**, como tipo, variedade de produtos e distância até a concorrência;
* **Informações sobre promoções**, incluindo participação em programas promocionais e períodos de promoção;
* **Informações temporais**, utilizadas para identificar padrões de sazonalidade, tendências e comportamento das vendas ao longo do tempo.

A solução deve considerar aspectos como:

* tratamento e qualidade dos dados;
* criação e seleção de variáveis;
* construção de uma **Feature Store por loja e data de referência**;
* criação de variáveis históricas, temporais, de sazonalidade e tendência;
* separação adequada entre treino, validação e teste, respeitando a ordem temporal dos dados;
* avaliação do desempenho do modelo;
* interpretação dos resultados;
* comparação entre as previsões do modelo e diferentes abordagens de referência (*baselines*).

A saída final do modelo deverá conter **o valor previsto de vendas para as próximas seis semanas de cada loja**, representando a soma estimada das vendas durante os 42 dias seguintes à data de referência.

> **Observação:** o horizonte de previsão utilizado neste projeto é de **42 dias (seis semanas)**. O modelo não tem como objetivo prever individualmente as vendas de cada dia, mas sim o **valor total de vendas esperado para cada loja ao longo das seis semanas seguintes**.


## Base de Dados

O projeto disponibiliza **duas bases de dados** contendo informações históricas de vendas e características das lojas de uma rede de farmácias. As bases representam o comportamento diário das lojas e suas principais características estruturais e promocionais.

As tabelas se relacionam principalmente por meio de:

* **Store**: identifica unicamente cada loja;
* **Date**: representa a data de referência das informações diárias de cada loja.

### Bases disponíveis

| Base    | Descrição                                                                                        | Granularidade |
| ------- | ------------------------------------------------------------------------------------------------ | ------------- |
| `sales` | Informações diárias sobre vendas, clientes, funcionamento, promoções e feriados.                 | Loja × dia    |
| `store` | Características estruturais e informações sobre concorrência e programas promocionais das lojas. | Loja          |

### Papel de cada base

A **sales** contém o histórico diário de cada loja, incluindo o número de vendas, quantidade de clientes, funcionamento da loja, promoções e feriados. Essa base é utilizada como principal fonte para a criação das variáveis históricas, temporais, de sazonalidade e tendência.

A **store** contém informações mais estruturais sobre cada loja, como tipo, variedade de produtos, distância até a concorrência e participação em programas promocionais. Essas informações são associadas à base `sales` por meio do identificador da loja.

A partir da combinação das duas bases, é construída uma **Feature Store por loja e data de referência**, utilizada para gerar as informações necessárias ao modelo preditivo.


## Sales

A base **sales** reúne informações históricas diárias sobre as lojas de uma rede de farmácias. Cada registro representa uma determinada loja em uma data específica, permitindo acompanhar o comportamento de vendas, fluxo de clientes, funcionamento, promoções e ocorrência de feriados ao longo do tempo.

A base possui **1.017.209 registros e 9 variáveis**.

Durante a exploração inicial, não foram identificados **valores ausentes** em nenhuma das variáveis. A principal atenção está relacionada à variável `Date`, que inicialmente apresenta o tipo `object` e deverá ser convertida para o formato de data para possibilitar a criação das variáveis temporais e das janelas históricas utilizadas no projeto.

## Dicionário de Dados

| Variável        | Descrição                           | Tipo    | Observações                                                                            |
| --------------- | ----------------------------------- | ------- | -------------------------------------------------------------------------------------- |
| `Store`         | Identificador da loja               | Inteiro | Utilizado como chave para relacionamento com a base `store`                            |
| `DayOfWeek`     | Dia da semana                       | Inteiro | Representa o dia da semana da observação                                               |
| `Date`          | Data da observação                  | Data    | Inicialmente armazenada como `object`; utilizada na construção das variáveis temporais |
| `Sales`         | Valor das vendas realizadas no dia  | Inteiro | Principal variável histórica utilizada para a previsão                                 |
| `Customers`     | Número de clientes atendidos no dia | Inteiro | Utilizada na criação de variáveis relacionadas ao fluxo de clientes                    |
| `Open`          | Indicador de funcionamento da loja  | Inteiro | 1 = aberta; 0 = fechada                                                                |
| `Promo`         | Indicador de promoção               | Inteiro | 1 = com promoção; 0 = sem promoção                                                     |
| `StateHoliday`  | Indicador de feriado estadual       | Texto   | Identifica a ocorrência de feriado estadual                                            |
| `SchoolHoliday` | Indicador de feriado escolar        | Inteiro | 1 = feriado escolar; 0 = sem feriado escolar                                           |

## Pontos observados

* A base possui **1.017.209 linhas e 9 colunas**, com granularidade diária por loja.
* Não foram identificados **valores ausentes** nas variáveis da base.
* A variável `Date` apresenta inicialmente o tipo `object` e deverá ser convertida para `datetime` durante o tratamento dos dados.
* `Sales` representa o volume diário de vendas e será utilizada tanto na construção das variáveis históricas quanto na definição do **target de previsão**.
* `Customers` representa o fluxo diário de clientes e será utilizada na criação de variáveis comportamentais.
* `Open`, `Promo`, `StateHoliday` e `SchoolHoliday` representam condições operacionais e eventos que podem influenciar o comportamento das vendas.
* `Store` será utilizada como chave para relacionar a base `sales` às características estruturais presentes na base `store`.
* A variável `Date` será utilizada como referência para a construção das janelas históricas, variáveis de tendência e sazonalidade.
* Para evitar **data leakage**, as variáveis utilizadas como entrada do modelo deverão considerar apenas informações disponíveis até a respectiva data de referência.


In [ ]:

# Conexão com o banco de dados
conn = sqlite3.connect('../../../data/database.db')

# Lista tabelas disponíveis no banco
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tabelas disponíveis no banco de dados:", tables['name'].tolist())

# Carrega a tabela de vendas
df_sales = pd.read_sql_query("SELECT * FROM sales", conn)

conn.close()

Tabelas disponíveis no banco de dados: ['store', 'sales']


In [ ]:
# Visualizar as primeiras linhas para conferência da estrutura do DataFrame de vendas
df_sales.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [ ]:
# Exibe o número de linhas e colunas do DataFrame de vendas
print(f"Número de linhas: {df_sales.shape[0]}")
print(f"Número de colunas: {df_sales.shape[1]}")

Número de linhas: 1017209
Número de colunas: 9


In [ ]:
# Conferir os tipos das variáveis da base de vendas
df_sales.dtypes

Store             int64
DayOfWeek         int64
Date             object
Sales             int64
Customers         int64
Open              int64
Promo             int64
StateHoliday     object
SchoolHoliday     int64
dtype: object

In [ ]:
# Verificando os valores únicos da coluna StateHoliday
print("Valores distintos de StateHoliday:", df_sales['StateHoliday'].unique())

Valores distintos de StateHoliday: ['0' 'a' 'b' 'c']


In [ ]:
# Calcula a porcentagem de valores ausentes por coluna
missing_percent = df_sales.isnull().mean() * 100

print("Porcentagem de valores ausentes por coluna:")
print(missing_percent)

Porcentagem de valores ausentes por coluna:
Store            0.0
DayOfWeek        0.0
Date             0.0
Sales            0.0
Customers        0.0
Open             0.0
Promo            0.0
StateHoliday     0.0
SchoolHoliday    0.0
dtype: float64


## Store

A base **store** reúne informações estruturais e características comerciais das lojas da rede de farmácias. Cada registro representa uma loja individual, identificada pela variável `Store`, e contém informações relacionadas ao tipo de loja, sortimento de produtos, concorrência e participação em programas promocionais.

A base possui **1.115 registros e 10 variáveis**.

Durante a exploração inicial, foram identificados valores ausentes principalmente nas informações relacionadas à **concorrência** e ao programa **Promo2**. A ausência de `CompetitionOpenSinceMonth` e `CompetitionOpenSinceYear` está relacionada às lojas que não possuem informações sobre a data de abertura da concorrência. Da mesma forma, `Promo2SinceWeek`, `Promo2SinceYear` e `PromoInterval` apresentam valores ausentes para lojas que não participam do programa Promo2.

## Dicionário de Dados

| Variável                    | Descrição                                             | Tipo       | Observações                                                     |
| --------------------------- | ----------------------------------------------------- | ---------- | --------------------------------------------------------------- |
| `Store`                     | Identificador da loja                                 | Inteiro    | Chave da base e utilizada no relacionamento com `train`         |
| `StoreType`                 | Tipo da loja                                          | Categórico | Classificação da loja entre os tipos A, B, C e D                |
| `Assortment`                | Tipo de sortimento                                    | Categórico | Classificação do nível de variedade de produtos da loja         |
| `CompetitionDistance`       | Distância até a concorrência mais próxima             | Numérico   | Aproximadamente 26,9% dos registros apresentam valores ausentes |
| `CompetitionOpenSinceMonth` | Mês de abertura da concorrência mais próxima          | Numérico   | Aproximadamente 31,7% dos registros apresentam valores ausentes |
| `CompetitionOpenSinceYear`  | Ano de abertura da concorrência mais próxima          | Numérico   | Aproximadamente 31,7% dos registros apresentam valores ausentes |
| `Promo2`                    | Indicador de participação no programa Promo2          | Inteiro    | 1 = participa; 0 = não participa                                |
| `Promo2SinceWeek`           | Semana em que a loja iniciou a participação no Promo2 | Numérico   | Aproximadamente 48,8% dos registros apresentam valores ausentes |
| `Promo2SinceYear`           | Ano em que a loja iniciou a participação no Promo2    | Numérico   | Aproximadamente 48,8% dos registros apresentam valores ausentes |
| `PromoInterval`             | Intervalo de meses em que a Promo2 é realizada        | Categórico | Aproximadamente 48,8% dos registros apresentam valores ausentes |

## Pontos observados

* A base possui **1.115 linhas e 10 colunas**, com granularidade de uma observação por loja.
* `Store` apresenta **100% de preenchimento** e será utilizada como chave para relacionar a base `store` à base `train`.
* `StoreType` e `Assortment` não apresentam valores ausentes e representam características estruturais das lojas.
* `CompetitionDistance` apresenta aproximadamente **26,9% de valores ausentes**, indicando que não há informação disponível sobre a distância até a concorrência para parte das lojas.
* `CompetitionOpenSinceMonth` e `CompetitionOpenSinceYear` apresentam aproximadamente **31,7% de valores ausentes**, relacionados à ausência de informações sobre a data de abertura da concorrência.
* As variáveis `Promo2SinceWeek`, `Promo2SinceYear` e `PromoInterval` apresentam aproximadamente **48,8% de valores ausentes**, principalmente porque nem todas as lojas participam do programa Promo2.
* As informações de competição e Promo2 serão utilizadas posteriormente na construção de variáveis temporais, como **tempo desde o início da competição** e **tempo desde o início da Promo2**, além dos indicadores de atividade desses eventos.
* O tratamento dos valores ausentes deverá considerar o significado de cada variável, evitando assumir que todos os valores ausentes representam o mesmo comportamento.


In [ ]:
# Conectar ao banco de dados SQLite
conn = sqlite3.connect('../../../data/database.db')

# Checar as tabelas disponíveis
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)
print("Tabelas disponíveis no banco de dados:", tables['name'].tolist())

# Ler a tabela 'store'
df_store = pd.read_sql_query("SELECT * FROM store", conn)

conn.close()  # Fechar conexão é importante

Tabelas disponíveis no banco de dados: ['store', 'sales']


In [ ]:
# Visualiza as primeiras linhas da base de lojas
df_store.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,None
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,None
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,None


In [ ]:
# Exibe o número de linhas e colunas do DataFrame de lojas
num_linhas = df_store.shape[0]
num_colunas = df_store.shape[1]

print(f"Número de linhas: {num_linhas}")
print(f"Número de colunas: {num_colunas}")

Número de linhas: 1115
Número de colunas: 10


In [ ]:
# Tipos de dados das colunas da base de lojas
df_store.dtypes

Store                          int64
StoreType                     object
Assortment                    object
CompetitionDistance          float64
CompetitionOpenSinceMonth    float64
CompetitionOpenSinceYear     float64
Promo2                         int64
Promo2SinceWeek              float64
Promo2SinceYear              float64
PromoInterval                 object
dtype: object

In [ ]:
# Calcula o percentual de valores ausentes por coluna
missing_percent = df_store.isnull().mean() * 100

print("Porcentagem de valores ausentes por coluna:")
print(missing_percent)

Porcentagem de valores ausentes por coluna:
Store                         0.000000
StoreType                     0.000000
Assortment                    0.000000
CompetitionDistance           0.269058
CompetitionOpenSinceMonth    31.748879
CompetitionOpenSinceYear     31.748879
Promo2                        0.000000
Promo2SinceWeek              48.789238
Promo2SinceYear              48.789238
PromoInterval                48.789238
dtype: float64
